# 03 — Load + embed

Scan `data/raw/<provider>/`, dispatch each file through the existing `Source` ingester, MERGE into Neo4j, then chunk + embed every Provision into the vector index.

Pick providers via `PROVIDERS` below. Files must already be downloaded by notebook 01 (`clg ingest ...`).

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

RAW = Path('data/raw')
PROVIDERS = ['eurlex', 'retsinformation']

for p in PROVIDERS:
    d = RAW / p
    n = sum(1 for _ in d.glob('*.xml')) if d.exists() else 0
    print(f'  {p:<20} {n} xml files in {d}')

In [ ]:
from crimellm.clg.config import get_settings
from crimellm.clg.graph import get_store

get_settings.cache_clear()
store = get_store()
store.verify()
print('connected to', store.settings.neo4j_uri)
print('embedding model:', store.settings.embedding_model)
print('embedding dim  :', store.settings.embedding_dim)

## Step 1 — clean slate (optional)

Drop existing Instruments/Provisions/Cases/Chunks so this notebook is idempotent. Schema (constraints + indexes) survives.

In [ ]:
with store.session() as s:
    s.run('MATCH (n) WHERE n:Instrument OR n:Provision OR n:Case OR n:Chunk DETACH DELETE n')
rows = store.run('MATCH (n) RETURN labels(n) AS lbls, count(*) AS n')
print('after wipe:', rows)

## Step 2 — discover per-provider IDs from filenames

Each provider's `Source` already knows how to parse + MERGE — we just feed it the IDs found on disk.

In [ ]:
import re
from collections import defaultdict

_EURLEX_RE = re.compile(r'^(?P<celex>[^.]+)\.(?P<lang>[a-z]{2,3})\.(?P<fmt>[a-z0-9_]+)\.xml$')
_RETSINFO_RE = re.compile(r'^([A-Z]\d{11})\.xml$')
_UK_RE = re.compile(r'^(?P<type>[a-z]+)-(?P<year>\d{4})-(?P<num>\d+)-(?P<version>[a-z0-9-]+)\.xml$')


def discover_eurlex(d: Path):
    """-> dict[fmt, (celex_ids, langs)]. EurLexSource takes one fmt at a time."""
    by_fmt: dict[str, dict[str, set[str]]] = defaultdict(lambda: {'celex': set(), 'langs': set()})
    for p in sorted(d.glob('*.xml')):
        m = _EURLEX_RE.match(p.name)
        if not m:
            print(f'  skip (unrecognised): {p.name}')
            continue
        by_fmt[m['fmt']]['celex'].add(m['celex'])
        by_fmt[m['fmt']]['langs'].add(m['lang'])
    return {fmt: (tuple(sorted(v['celex'])), tuple(sorted(v['langs']))) for fmt, v in by_fmt.items()}


def discover_retsinformation(d: Path):
    accns = []
    for p in sorted(d.glob('*.xml')):
        m = _RETSINFO_RE.match(p.name)
        if not m:
            print(f'  skip (unrecognised): {p.name}')
            continue
        accns.append(m.group(1))
    return tuple(accns)


def discover_legislation_uk(d: Path):
    statutes: set[tuple[str, int, int]] = set()
    versions: set[str] = set()
    for p in sorted(d.glob('*.xml')):
        m = _UK_RE.match(p.name)
        if not m:
            print(f'  skip (unrecognised): {p.name}')
            continue
        statutes.add((m['type'], int(m['year']), int(m['num'])))
        versions.add(m['version'])
    return tuple(sorted(statutes)), tuple(sorted(versions))


if 'eurlex' in PROVIDERS:
    print('eurlex:')
    for fmt, (celex, langs) in discover_eurlex(RAW / 'eurlex').items():
        print(f'  fmt={fmt:<12} langs={langs} celex={len(celex)} -> {list(celex)[:3]}{"..." if len(celex) > 3 else ""}')

if 'retsinformation' in PROVIDERS:
    accns = discover_retsinformation(RAW / 'retsinformation')
    print(f'retsinformation: {len(accns)} accns -> {list(accns)[:5]}{"..." if len(accns) > 5 else ""}')

if 'legislation_uk' in PROVIDERS:
    statutes, versions = discover_legislation_uk(RAW / 'legislation_uk')
    print(f'legislation_uk: {len(statutes)} statutes × {len(versions)} versions={versions}')

## Step 3 — dispatch each provider through its `Source.load(ctx)`

Files already on disk — each `Source.download(...)` is a no-op (cached). `Source.load(...)` parses + MERGEs Instruments / Provisions / Cases / IMPLEMENTS in one shot and returns counts.

In [ ]:
from crimellm.clg.ingest._base import IngestContext

ctx = IngestContext(store=store)
reports = []


def _safe_load(label, src):
    try:
        rep = src.load(ctx)
    except Exception as e:
        print(f'{label}: FAILED -> {type(e).__name__}: {e}')
        return
    reports.append(rep)
    print(f'{label}:', rep.counts)


if 'eurlex' in PROVIDERS:
    from crimellm.clg.ingest.eurlex import EurLexSource
    for fmt, (celex, langs) in discover_eurlex(RAW / 'eurlex').items():
        if not celex:
            continue
        _safe_load(f'eurlex[{fmt}]', EurLexSource(celex_ids=celex, languages=langs, fmt=fmt))

if 'retsinformation' in PROVIDERS:
    from crimellm.clg.ingest.retsinformation import RetsinformationSource
    accns = discover_retsinformation(RAW / 'retsinformation')
    if accns:
        _safe_load('retsinformation', RetsinformationSource(accns=accns))

if 'legislation_uk' in PROVIDERS:
    from crimellm.clg.ingest.legislation_uk import LegislationUKSource
    statutes, versions = discover_legislation_uk(RAW / 'legislation_uk')
    if statutes:
        _safe_load('legislation_uk', LegislationUKSource(statutes=statutes, versions=versions))

print()
print('totals by label after load:')
rows = store.run(
    'MATCH (n) UNWIND labels(n) AS l RETURN l AS label, count(*) AS n ORDER BY label'
)
for r in rows:
    print(f"  {r['label']:<14} {r['n']:>6}")

## Step 4 — chunk + embed every Provision

Pull all Provisions from Neo4j and embed them in one pass. Vector index is rebuilt if its dim disagrees with the embedder.

In [ ]:
from crimellm.clg.embed.chunker import chunk_provision
from crimellm.clg.embed.embedder import get_embedder
from crimellm.clg.graph.loaders import load_chunks, vector_index_dim
from crimellm.clg.graph.schema import rebuild_vector_index
from crimellm.clg.models import Provision

embedder = get_embedder('sentence-transformers', model='Qwen/Qwen3-Embedding-0.6B')
print(f'embedder: {embedder.name} (dim={embedder.dim})')

current = vector_index_dim(store)
if current != embedder.dim:
    print(f'rebuilding vector index from {current} -> {embedder.dim}')
    rebuild_vector_index(embedder.dim, drop_chunks=True, store=store)

rows = store.run(
    'MATCH (p:Provision) RETURN p.id AS id, p.instrument_id AS instrument_id, '
    'p.jurisdiction AS jurisdiction, p.section_path AS section_path, p.text AS text'
)
provisions = [
    Provision(
        id=r['id'],
        instrument_id=r['instrument_id'],
        jurisdiction=r['jurisdiction'],
        section_path=r['section_path'] or '',
        text=r['text'] or '',
    )
    for r in rows
]
print(f'provisions in DB: {len(provisions)}')

chunks = [c for p in provisions for c in chunk_provision(p)]
print(f'chunks to embed: {len(chunks)}')

if chunks:
    vecs = embedder.embed_batch([c.text for c in chunks])
    for c, v in zip(chunks, vecs, strict=True):
        c.embedding = v
    n = load_chunks(chunks, embedding_model=embedder.name, store=store)
    print(f'loaded {n} chunks')

## Step 5 — verify the vector index resolves up to entity

Pick a chunk, embed its own text, search → top hit should be the same Provision (cosine ≈ 1.0 for a real encoder on identical text).

In [ ]:
from crimellm.clg.graph.loaders import search_chunks

if chunks:
    seed = chunks[min(2, len(chunks) - 1)]
    qvec = embedder.embed(seed.text)
    hits = search_chunks(qvec, k=3, store=store)
    for h in hits:
        print(
            f"{h['score']:.3f} {h['parent_jurisdiction']:<3} "
            f"{h['parent_type']:<10} {h['parent_id']}"
        )

## Summary

In [ ]:
rows = store.run(
    'MATCH (n) UNWIND labels(n) AS l RETURN l AS label, count(*) AS n ORDER BY label'
)
for r in rows:
    print(f"  {r['label']:<14} {r['n']:>6}")

rels = store.run(
    'MATCH ()-[r]->() RETURN type(r) AS t, count(*) AS n ORDER BY t'
)
print()
for r in rels:
    print(f"  {r['t']:<14} {r['n']:>6}")

## Next: notebook 04 — synthesize a grounded answer

[`04_synthesize_query.ipynb`](04_synthesize_query.ipynb) runs end-to-end queries: EN + DA prompts, jurisdiction inference, cross-jurisdiction IMPLEMENTS traversal, audit-friendly answer metadata.